In [10]:
# XAI PROJECT: Loan Approval Model
# Logistic Regression + SHAP + DiCE

# Imports 
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

import shap
import dice_ml
from dice_ml import Dice

# Reproducibility
np.random.seed(42)

plt.style.use("ggplot")


# Load Dataset
data = pd.read_csv("adult.csv")

print("Dataset preview:")
print(data.head())


# Preprocessing 

# Handle missing values
data = data.replace("?", np.nan)
data = data.dropna()

# Convert target to binary
data['income'] = data['income'].apply(lambda x: 1 if '>50K' in x else 0)

# Split features and target
X = data.drop("income", axis=1)
y = data["income"]

# One-hot encoding
X = pd.get_dummies(X, drop_first=True)

# convert all to numeric (no True/False issues)
X = X.astype(float)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# Model (Pipeline) 

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

print("\nModel Accuracy:", accuracy_score(y_test, y_pred))


# Create results folder
os.makedirs("results", exist_ok=True)


# SHAP EXPLANATIONS 

# Extract scaler + model
scaler = pipeline.named_steps["scaler"]
model = pipeline.named_steps["model"]

# Scale data for SHAP
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# SHAP LinearExplainer (correct for logistic regression)
explainer = shap.LinearExplainer(model, X_train_scaled)

shap_values = explainer(X_test_scaled)

# Global SHAP 
plt.figure()
shap.summary_plot(shap_values, X_test, show=False)
plt.savefig("results/shap_summary.png")
plt.close()

# Local SHAP
plt.figure()
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[0],
        base_values=explainer.expected_value,
        data=X_test.iloc[0],
        feature_names=X_test.columns
    ),
    show=False
)
plt.savefig("results/shap_waterfall.png")
plt.close()


# DiCE COUNTERFACTUALS 

# Combine for DiCE
df_dice = pd.concat([X, y], axis=1)

# Automatically detect continuous features
continuous_features = X.select_dtypes(include=['float64', 'int64']).columns.tolist()

# Remove binary features (keep real continuous only)
continuous_features = [col for col in continuous_features if X[col].nunique() > 2]

# Define categorical features
categorical_features = [col for col in X.columns if col not in continuous_features]

print("\nContinuous features:", continuous_features[:5])
print("Categorical features:", categorical_features[:5])

# Create DiCE data object
dice_data = dice_ml.Data(
    dataframe=df_dice,
    continuous_features=continuous_features,
    categorical_features=categorical_features,
    outcome_name='income'
)

# Use pipeline (correct scaling)
dice_model = dice_ml.Model(
    model=pipeline,
    backend="sklearn"
)

dice_exp = Dice(dice_data, dice_model)

# Generate Counterfactuals 
success_count = 0

for i in range(10):
    try:
        query_instance = X_test.iloc[i:i+1]

        counterfactuals = dice_exp.generate_counterfactuals(
    query_instance,
    total_CFs=2,
    desired_class="opposite",
    features_to_vary=['hours-per-week', 'capital-gain', 'educational-num'],


    permitted_range={
        'hours-per-week': [20, 60],
        'educational-num': [8, 16],
        'capital-gain': [0, 5000]
    }
)

        print(f"\nCounterfactuals found for index {i}")
        counterfactuals.visualize_as_dataframe()

        success_count += 1

    except Exception as e:
        print(f"Failed for index {i}: {e}")

print("\nCounterfactual success rate:", success_count / 10)


# BIAS CHECK 


if 'sex_Male' in X_test.columns:

    male_mask = X_test['sex_Male'] == 1
    female_mask = X_test['sex_Male'] == 0

    male_preds = pipeline.predict(X_test[male_mask])
    female_preds = pipeline.predict(X_test[female_mask])

    male_rate = np.mean(male_preds)
    female_rate = np.mean(female_preds)

    print("\nBias check:")
    print("Male approval rate:", male_rate)
    print("Female approval rate:", female_rate)
    print("Difference:", abs(male_rate - female_rate))



Dataset preview:
   age  workclass  fnlwgt     education  educational-num      marital-status  \
0   25    Private  226802          11th                7       Never-married   
1   38    Private   89814       HS-grad                9  Married-civ-spouse   
2   28  Local-gov  336951    Assoc-acdm               12  Married-civ-spouse   
3   44    Private  160323  Some-college               10  Married-civ-spouse   
4   18          ?  103497  Some-college               10       Never-married   

          occupation relationship   race  gender  capital-gain  capital-loss  \
0  Machine-op-inspct    Own-child  Black    Male             0             0   
1    Farming-fishing      Husband  White    Male             0             0   
2    Protective-serv      Husband  White    Male             0             0   
3  Machine-op-inspct      Husband  Black    Male          7688             0   
4                  ?    Own-child  White  Female             0             0   

   hours-per-week nat

/var/folders/r4/47cydfbd2hjc5g2s93xhkrvc0000gn/T/ipykernel_2472/1870301038.py:93: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, X_test, show=False)



Continuous features: ['age', 'fnlwgt', 'educational-num', 'capital-gain', 'capital-loss']
Categorical features: ['workclass_Local-gov', 'workclass_Private', 'workclass_Self-emp-inc', 'workclass_Self-emp-not-inc', 'workclass_State-gov']


100%|██████████| 1/1 [00:00<00:00,  7.33it/s]


No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
Failed for index 0: No counterfactuals found for any of the query points! Kindly check your configuration.


...


Counterfactuals found for index 9
Query instance (original outcome : 0)


,age,fnlwgt,educational-num,capital-gain,capital-loss,hours-per-week,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,...,native-country_Puerto-Rico,native-country_Scotland,native-country_South,native-country_Taiwan,native-country_Thailand,native-country_Trinadad&Tobago,native-country_United-States,native-country_Vietnam,native-country_Yugoslavia,income
0,29.0,327779.0,10.0,0.0,0.0,20.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0



Diverse Counterfactual set (new outcome: 1)


,age,fnlwgt,educational-num,capital-gain,capital-loss,hours-per-week,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,...,native-country_Puerto-Rico,native-country_Scotland,native-country_South,native-country_Taiwan,native-country_Thailand,native-country_Trinadad&Tobago,native-country_United-States,native-country_Vietnam,native-country_Yugoslavia,income
0,29.0,327779.0,15.8,3827.4,0.0,20.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1
1,29.0,327779.0,15.2,4388.6,0.0,20.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1



Counterfactual success rate: 0.8
